# Convert PyTorch checkpoints to HDF5

This converts all six GAN/VAE `.pt` state dictionaries into genuine HDF5 files beside their source checkpoints under `models/part_a/improved/` and `models/part_a/others/`. The weights remain PyTorch state dictionaries; changing the container to `.h5` does not turn them into Keras models.

In [ ]:
# Run this once if h5py is not installed in the current notebook kernel.
%pip install -q h5py

In [ ]:
from pathlib import Path

import h5py
import torch

checkpoint_files = [
    Path("models/part_a/others/gan_baseline_generator.pt"),
    Path("models/part_a/others/gan_baseline_discriminator.pt"),
    Path("models/part_a/improved/best_gan.pt"),
    Path("models/part_a/others/vae_baseline_best.pt"),
    Path("models/part_a/others/vae_classifier.pt"),
    Path("models/part_a/improved/vae_final_best.pt"),
]

In [ ]:
def convert_pt_to_h5(pt_path):
    """Copy one PyTorch state dictionary into an HDF5 file."""
    state_dict = torch.load(pt_path, map_location="cpu", weights_only=True)
    h5_path = pt_path.with_suffix(".h5")

    with h5py.File(h5_path, "w") as h5_file:
        h5_file.attrs["framework"] = "pytorch"
        h5_file.attrs["format"] = "state_dict"
        h5_file.attrs["source_file"] = pt_path.name
        weights = h5_file.create_group("state_dict")

        for name, tensor in state_dict.items():
            weights.create_dataset(name, data=tensor.detach().cpu().numpy())

    return h5_path


for checkpoint in checkpoint_files:
    if not checkpoint.exists():
        print(f"Skipped missing file: {checkpoint}")
        continue

    converted = convert_pt_to_h5(checkpoint)
    with h5py.File(converted, "r") as h5_file:
        tensor_count = len(h5_file["state_dict"])
    print(f"Created {converted} ({tensor_count} tensors)")